# mBERT Fine-Tuning

`bert-base-multilingual-cased`, fine-tuned end-to-end for binary sequence classification.

**Methodology (different from the RNN/CNN notebooks, due to compute cost):**
- Hyperparameter search: **single held-out split** (fold 1 of `StratifiedGroupKFold`), not mean-across-5-folds — full 5-fold search is impractical at transformer scale.
- Final reporting: **5-fold cross-validation** with the best config, plus a holdout test evaluation — same as the RNN/CNN notebooks, so results stay directly comparable.
- Uses `AdamW` + linear warmup/decay schedule, low learning rates (2e-5 to 5e-5 range), and gradient clipping (standard practice for transformer fine-tuning, unlike the CNN notebook).
- WordPiece subword tokenization (not the whitespace tokenizer used for RNN/CNN) - `max_length` is set based on tokenized length, not raw word count.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import warnings
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
load_dotenv()

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, classification_report
)
import wandb

# ── Reproducibility ───────────────────────────────────────────────
RANDOM_SEED = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODEL_NAME = 'bert-base-multilingual-cased'

print('Libraries loaded.')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')

In [ ]:
wandb_key = os.environ.get('WANDB_API_KEY')
if not wandb_key:
    raise ValueError('Set WANDB_API_KEY in your .env file')

wandb.login(key=wandb_key)
wandb.init(
    project = 'commitment-mining',
    name    = 'dl-mbert-finetune-aug',
    config  = {'model': MODEL_NAME, 'framework': 'pytorch/transformers'},
    tags    = ['mbert', 'transformer', 'deep-learning', 'pytorch', 'augmented']
)
print('WandB initialized.')

In [ ]:
TRAIN_PATH = 'Commitment-Mining/dataset/train_80p-aug.xlsx'
TEST_PATH  = 'Commitment-Mining/dataset/test_20p.xlsx'
TEXT_COL   = 'statements (ne)'
LABEL_COL  = 'final_label'

train_df = pd.read_excel(TRAIN_PATH)
test_df  = pd.read_excel(TEST_PATH)

for df in [train_df, test_df]:
    df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

print(f'Train size : {len(train_df)}')
print(f'Test size  : {len(test_df)}')
print('\nTrain label distribution:')
print(train_df[LABEL_COL].value_counts())

label_counts = train_df[LABEL_COL].value_counts().to_dict()
wandb.log({'train_size': len(train_df), 'test_size': len(test_df), **label_counts})

In [ ]:
# Same stratified-group split key as the RNN/CNN/ML notebooks
train_df['sentence_length'] = train_df[TEXT_COL].apply(
    lambda x: pd.cut(
        [len(x.split())],
        bins=[0, 5, 10, 20, 50, 999],
        labels=['xs', 's', 'm', 'l', 'xl']
    )[0]
)

train_df['strat_key'] = (
    train_df['province'].astype(str)           + '_' +
    train_df['sentence_length'].astype(str)    + '_' +
    train_df['district/gaupalika'].astype(str) + '_' +
    train_df[LABEL_COL].astype(str)
)

counts = train_df['strat_key'].value_counts()
rare   = counts[counts < 5].index
train_df['strat_key'] = train_df['strat_key'].apply(
    lambda x: 'rare' if x in rare else x
)

le      = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL].values)
y_test  = le.transform(test_df[LABEL_COL].values)
groups  = train_df['strat_key'].values

X_train_text = [str(x) for x in train_df[TEXT_COL].tolist()]
X_test_text  = [str(x) for x in test_df[TEXT_COL].tolist()]

print(f'Classes  : {le.classes_}')
print(f'Unique strat keys : {train_df["strat_key"].nunique()}')

## Tokenization - WordPiece subwords (not whitespace splitting)

Check actual tokenized lengths first, since mBERT's subword tokenizer can split a single Nepali word into multiple tokens — `max_length` needs to be based on this, not the raw word count used for the RNN/CNN notebooks.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Check the real subword-token length distribution before picking MAX_LEN
sample_lengths = [len(tokenizer.encode(t, add_special_tokens=True)) for t in X_train_text]
print(f'Mean subword length   : {np.mean(sample_lengths):.1f}')
print(f'Median subword length : {np.median(sample_lengths):.1f}')
print(f'95th percentile       : {np.percentile(sample_lengths, 95):.1f}')
print(f'Max subword length    : {max(sample_lengths)}')

MAX_LEN = 128  # adjust based on the printed distribution above if needed
print(f'\nUsing MAX_LEN = {MAX_LEN}')
print(f'% sequences truncated at this MAX_LEN: '
      f'{sum(l > MAX_LEN for l in sample_lengths) / len(sample_lengths) * 100:.2f}%')

In [ ]:
class BertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

## Model, training, and prediction functions

In [ ]:
def build_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=1, problem_type='regression'  # 1 logit + BCEWithLogitsLoss, same convention as RNN/CNN notebooks
    ).to(DEVICE)
    return model


def train_model(model, train_texts, train_labels, val_texts, val_labels, config,
                 epochs=10, patience=3, verbose=1):
    """
    Standard transformer fine-tuning loop:
    AdamW + linear warmup/decay, gradient clipping, early stopping on val loss.
    """
    train_ds = BertDataset(train_texts, train_labels, tokenizer, MAX_LEN)
    val_ds   = BertDataset(val_texts, val_labels, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'],
                                   weight_decay=config['weight_decay'])

    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    criterion = nn.BCEWithLogitsLoss()

    best_val_loss = float('inf')
    best_state = None
    patience_ctr = 0
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        running_loss, n_seen = 0.0, 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze(-1)
            loss = criterion(logits, labels)
            loss.backward()

            # Gradient clipping — standard practice for transformer fine-tuning
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            running_loss += loss.item() * len(labels)
            n_seen += len(labels)
        train_loss = running_loss / n_seen

        model.eval()
        val_running_loss, n_val = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['label'].to(DEVICE)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits.squeeze(-1)
                loss = criterion(logits, labels)
                val_running_loss += loss.item() * len(labels)
                n_val += len(labels)
        val_loss = val_running_loss / n_val

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if verbose:
            print(f'  Epoch {epoch+1}/{epochs} | loss: {train_loss:.4f} | val_loss: {val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return train_losses, val_losses


def predict_proba(model, texts, batch_size=32):
    ds = BertDataset(texts, [0] * len(texts), tokenizer, MAX_LEN)  # dummy labels, unused
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze(-1)
            probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs)


def compute_metrics(y_true, y_pred, y_proba):
    return {
        'accuracy'   : accuracy_score(y_true, y_pred),
        'precision'  : precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall'     : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1'   : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'auroc'      : roc_auc_score(y_true, y_proba)
    }

## Hyperparameter search 

In [ ]:
param_distributions = {
    'learning_rate' : [2e-5, 3e-5, 5e-5],
    'batch_size'    : [8, 16],
    'weight_decay'  : [0.0, 0.01],
}

N_ITER = 10   # small budget — each config is a full mBERT fine-tune run

random.seed(RANDOM_SEED)
sampled_configs = [
    {k: random.choice(v) for k, v in param_distributions.items()}
    for _ in range(N_ITER)
]
print(f'Total configs to try : {N_ITER}')
print('Sample config 1:', sampled_configs[0])

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
splits = list(sgkf.split(X_train_text, y_train, groups=groups))

# Fold 1 only, for hyperparameter search
tr_idx_rs, val_idx_rs = splits[0]

X_tr_rs = [X_train_text[i] for i in tr_idx_rs]
y_tr_rs = y_train[tr_idx_rs]
X_val_rs = [X_train_text[i] for i in val_idx_rs]
y_val_rs = y_train[val_idx_rs]

print(f'Search train : {len(X_tr_rs)}')
print(f'Search val   : {len(X_val_rs)}')

search_results = []

for i, config in enumerate(sampled_configs):
    print(f'\n--- Config {i+1}/{N_ITER}: {config} ---')
    torch.manual_seed(RANDOM_SEED)
    model = build_model()

    train_model(model, X_tr_rs, y_tr_rs, X_val_rs, y_val_rs, config,
                epochs=10, patience=3, verbose=1)

    y_proba_val = predict_proba(model, X_val_rs)
    y_pred_val  = (y_proba_val > 0.5).astype(int)
    val_f1      = f1_score(y_val_rs, y_pred_val, average='macro', zero_division=0)

    search_results.append({'config': config, 'val_f1': val_f1})
    print(f'Config {i+1}/{N_ITER} | val_f1 (fold 1 only): {val_f1:.4f}')

    wandb.log({
        f'search/config_{i+1}_val_f1': val_f1,
        **{f'search/config_{i+1}_{k}': v for k, v in config.items()}
    })

    del model
    torch.cuda.empty_cache()

best_result = max(search_results, key=lambda x: x['val_f1'])
best_config = best_result['config']

print(f'\n✓ Search complete (single-split).')
print(f'Best val F1 : {best_result["val_f1"]:.4f}')
print(f'Best config : {best_config}')

wandb.config.update({
    'best_search_val_f1': best_result['val_f1'],
    **{f'best_param/{k}': v for k, v in best_config.items()}
})

## 5-fold cross-validation with best config

In [ ]:
fold_metrics = []
all_histories = []

for fold, (tr_idx, val_idx) in enumerate(splits):
    X_tr = [X_train_text[i] for i in tr_idx]
    y_tr = y_train[tr_idx]
    X_val = [X_train_text[i] for i in val_idx]
    y_val = y_train[val_idx]

    torch.manual_seed(RANDOM_SEED)
    model = build_model()

    train_loss_hist, val_loss_hist = train_model(
        model, X_tr, y_tr, X_val, y_val, best_config,
        epochs=10, patience=3, verbose=0
    )
    all_histories.append((train_loss_hist, val_loss_hist))

    y_proba = predict_proba(model, X_val)
    y_pred  = (y_proba > 0.5).astype(int)

    m = compute_metrics(y_val, y_pred, y_proba)
    m['fold'] = fold + 1
    fold_metrics.append(m)

    print(f"Fold {fold+1} | "
          f"Acc: {m['accuracy']:.4f} | "
          f"F1(w): {m['f1_weighted']:.4f} | "
          f"F1(macro): {m['f1']:.4f} | "
          f"AUROC: {m['auroc']:.4f}")

    del model
    torch.cuda.empty_cache()

for fold, (train_loss, val_loss) in enumerate(all_histories):
    for epoch, (tl, vl) in enumerate(zip(train_loss, val_loss)):
        wandb.log({
            f'fold_{fold+1}/train_loss': tl,
            f'fold_{fold+1}/val_loss'  : vl,
            f'fold_{fold+1}/epoch'     : epoch
        })

fold_df    = pd.DataFrame(fold_metrics).set_index('fold')
mean_row   = fold_df.mean().rename('mean')
std_row    = fold_df.std().rename('std')
cv_summary = pd.concat([fold_df, mean_row.to_frame().T, std_row.to_frame().T])

print('\n=== CV Results (best config) ===')
print(cv_summary.round(4))

wandb.log({
    'cv/accuracy_mean'    : fold_df['accuracy'].mean(),
    'cv/accuracy_std'     : fold_df['accuracy'].std(),
    'cv/f1_weighted_mean' : fold_df['f1_weighted'].mean(),
    'cv/f1_weighted_std'  : fold_df['f1_weighted'].std(),
    'cv/f1_macro_mean'    : fold_df['f1'].mean(),
    'cv/f1_macro_std'     : fold_df['f1'].std(),
    'cv/auroc_mean'       : fold_df['auroc'].mean(),
    'cv/auroc_std'        : fold_df['auroc'].std(),
    'cv_results'          : wandb.Table(dataframe=cv_summary.round(4))
})

## Final holdout test evaluation

In [ ]:
X_tr_full, X_val_full, y_tr_full, y_val_full = train_test_split(
    X_train_text, y_train,
    test_size    = 0.1,
    stratify     = y_train,
    random_state = RANDOM_SEED
)

torch.manual_seed(RANDOM_SEED)
final_model = build_model()

train_model(
    final_model, X_tr_full, y_tr_full, X_val_full, y_val_full, best_config,
    epochs=10, patience=3, verbose=1
)

y_proba_test = predict_proba(final_model, X_test_text)
y_pred_test  = (y_proba_test > 0.5).astype(int)

test_metrics = compute_metrics(y_test, y_pred_test, y_proba_test)
np.save('y_pred-mbert.npy', y_pred_test)
np.save('y_true-mbert.npy', y_test)

print('=== HOLDOUT TEST SET RESULTS ===')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_test, target_names=le.classes_))

wandb.log({f'test/{k}': v for k, v in test_metrics.items()})

report_df = pd.DataFrame(
    classification_report(y_test, y_pred_test,
                          target_names=le.classes_, output_dict=True)
).transpose().round(4)
wandb.log({'classification_report': wandb.Table(dataframe=report_df)})

In [ ]:
metrics_order = ['accuracy', 'precision', 'recall', 'f1_weighted', 'f1', 'auroc']

summary = pd.DataFrame({
    'CV Mean' : fold_df[metrics_order].mean().round(4),
    'CV Std'  : fold_df[metrics_order].std().round(4),
    'Test'    : pd.Series(test_metrics)[metrics_order].round(4)
})

print('=== PAPER TABLE — mBERT (bert-base-multilingual-cased) ===')
print(summary)
print('\nBest hyperparameters:')
for k, v in best_config.items():
    print(f'  {k}: {v}')
print('\nNOTE: hyperparameter search used a single held-out split (fold 1),')
print('not mean-across-5-folds, due to compute cost. CV/test results above')
print('use proper 5-fold cross-validation with the selected best config.')

wandb.log({'paper_table': wandb.Table(dataframe=summary)})
wandb.finish()